In [1]:
# Import libraries
import pandas as pd
import re
import contractions

In [2]:
# Load dataset
df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/storage/englishverses.csv", encoding='utf-8-sig')
print("Initial shape:", df.shape)


Initial shape: (47431, 39)


In [3]:
def clean_lyrics(text):
    if not isinstance(text, str):
        return ""

    # 1. Remove square brackets and contents
    text = re.sub(r"\[.*?\]", "", text, flags=re.DOTALL)

    # 2. Normalize apostrophes
    text = text.replace("’", "'")

    # 3. Convert adlibs: (adlib) → , adlib,
    text = re.sub(r"\s*\((.*?)\)", r", \1,", text)

    # 4. Fix merged words
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)

    # 7. Apply contractions (easier for model training)
    try:
        text = contractions.fix(text)
    except:
        pass

    # 8. Lowercase everything
    text = text.lower()

    # 9. Remove double commas
    text = re.sub(r",\s*,+", ", ", text)

    # 10. Clean spaces around commas
    text = re.sub(r"\s+,", ",", text)
    text = re.sub(r",\s+", ", ", text)

    # 11. Remove trailing commas in each line
    text = re.sub(r",\s*$", "", text, flags=re.MULTILINE)

    return text.strip()

In [4]:
# Apply cleaning
df['lyrics'] = df['genius_lyrics'].apply(clean_lyrics)

In [5]:
# Remove songs that have "white noise", "sleep" in genre
df = df[~df['artist_genres'].str.contains("white noise", case=False, na=False)]
df = df[~df['artist_genres'].str.contains("sleep", case=False, na=False)]
df = df[~df['artist_genres'].str.contains("native american music", case=False, na=False)]
print("After removing:", df.shape)

After removing: (47043, 40)


In [6]:
# Remove genius_lyrics columns
df = df.drop(columns=['genius_lyrics'])

In [ ]:
# Save output
print("After cleaning:", df.shape)
df.to_csv('C:/Users/User/Documents/devanasokan_fyp/storage/cleanverses.csv', index=False, encoding='utf-8-sig')

After cleaning: (47043, 39)
